# Pipeline 2 Preprocessing: Graph Dataset Builder

Colab-ready preprocessing notebook for the **GNN + GRU pipeline only**.

This notebook processes states sequentially:

```text
Assam -> Kerala -> Uttarpradesh -> Odisha
```

It does **not** merge states and does **not** train GNN+GRU. For each state, it converts Pipeline 1 patch chunks into a graph-ready dataset:

```text
Pipeline 1 patch chunks + rainfall/river data
        -> node features
        -> graph edges
        -> flood targets
        -> hydro sequences
        -> per-state graph dataset folder
```


## 1. Install Dependencies


In [ ]:
!pip install -q pandas numpy scikit-learn tqdm


## 2. Mount Drive and Imports


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')


## 3. Configure Paths and State Order

Edit `DRIVE_ROOT`, `PATCH_ROOT`, and hydrological file paths if your Drive layout differs.

Expected patch input per state is the Pipeline 1 preprocessing output with per-year folders and `patches/` chunks.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/PRJ3_Data')
ALT_DRIVE_ROOTS = [
    Path('/content/drive/MyDrive/Prj_3_Data'),
    Path('/content/drive/MyDrive/PRJ3_DATA'),
    Path('/content/drive/MyDrive/Prj3_Data'),
]

OUTPUT_ROOT = DRIVE_ROOT / 'Pipeline2_Graph_Datasets'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STATE_ORDER = ['Assam', 'Kerala', 'Uttarpradesh', 'Odisha']
YEARS = [2019, 2020, 2021, 2022, 2023]
SEQ_LEN = 30
FLOOD_THRESHOLD = 0.01
NEIGHBOR_RADIUS = 1

# Candidate state-specific Pipeline 1 patch roots. The notebook uses the first existing path.
STATE_CONFIGS = {
    'Assam': {
        'patch_roots': [
            DRIVE_ROOT / 'Assam_Pipeline1_Model_Ready',
            DRIVE_ROOT / 'Assam_Model_Ready',
            DRIVE_ROOT / 'ASSAM_Pipeline1_Model_Ready',
        ],
        'rainfall_files': [DRIVE_ROOT / 'ASSAM_PRJ3' / f'Assam_{year}.csv' for year in YEARS],
        'river_files': [DRIVE_ROOT / 'ASSAM_PRJ3' / 'Assam_Riverlevel.csv', DRIVE_ROOT / 'Assam_Riverlevel.csv'],
    },
    'Kerala': {
        'patch_roots': [
            DRIVE_ROOT / 'Kerala_Pipeline1_Model_Ready',
            DRIVE_ROOT / 'Kerala_Model_Ready',
            DRIVE_ROOT / 'KERALA_Pipeline1_Model_Ready',
        ],
        'rainfall_files': [DRIVE_ROOT / 'KERALA_PRJ3' / f'Kerala_{year}.csv' for year in YEARS],
        'river_files': [DRIVE_ROOT / 'KERALA_PRJ3' / 'Kerala_Riverlevel.csv', DRIVE_ROOT / 'Kerala_Riverlevel.csv'],
    },
    'Uttarpradesh': {
        'patch_roots': [
            DRIVE_ROOT / 'Uttarpradesh_Pipeline1_Model_Ready',
            DRIVE_ROOT / 'Uttarpradesh_Model_Ready',
            DRIVE_ROOT / 'UTTARPRADESH_Pipeline1_Model_Ready',
        ],
        'rainfall_files': [DRIVE_ROOT / 'UTTARPRADESH_PRJ3' / f'Uttarpradesh_{year}.csv' for year in YEARS],
        'river_files': [DRIVE_ROOT / 'UTTARPRADESH_PRJ3' / 'Uttarpradesh_Riverlevel.csv', DRIVE_ROOT / 'Uttarpradesh_Riverlevel.csv'],
    },
    'Odisha': {
        'patch_roots': [
            DRIVE_ROOT / 'Odisha_Pipeline1_Model_Ready',
            DRIVE_ROOT / 'Odisha_Model_Ready',
            DRIVE_ROOT / 'ODISHA_Pipeline1_Model_Ready',
        ],
        'rainfall_files': [DRIVE_ROOT / 'ODISHA_PRJ3' / f'Odisha_{year}.csv' for year in YEARS],
        'river_files': [DRIVE_ROOT / 'ODISHA_PRJ3' / 'Odisha_Riverlevel.csv', DRIVE_ROOT / 'Odisha_Riverlevel.csv'],
    },
}

print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('State order:', STATE_ORDER)


## 4. Utility Functions

These functions are designed to support both newer Pipeline 1 names (`X_s1_pre_*`) and older names (`X_pre_*`).


In [ ]:


def expand_drive_candidates(paths):
    expanded = []
    for path in paths:
        path = Path(path)
        expanded.append(path)
        for root in ALT_DRIVE_ROOTS:
            # Preserve the final one or two path segments when trying alternate roots.
            expanded.append(root / path.name)
            if len(path.parts) >= 2:
                expanded.append(root / path.parts[-2] / path.name)
    seen = []
    for path in expanded:
        if path not in seen:
            seen.append(path)
    return seen

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


def resolve_patch_root(state):
    config = STATE_CONFIGS[state]
    candidates = []
    for root in ALT_DRIVE_ROOTS:
        for p in config['patch_roots']:
            candidates.append(root / p.name)
    candidates.extend(config['patch_roots'])
    resolved = first_existing(candidates)
    if resolved is None:
        print(f'{state}: no patch root found. Checked:')
        for p in candidates:
            print(' ', p)
    return resolved


def find_chunk_files(patch_dir, prefixes):
    files = []
    for prefix in prefixes:
        files.extend(sorted(patch_dir.glob(f'{prefix}_*.npy')))
    return sorted(set(files))


def chunk_id_from_name(path):
    nums = re.findall(r'(\d+)', path.stem)
    return int(nums[-1]) if nums else 0


def collect_year_chunks(patch_root, year):
    patch_dir = patch_root / str(year) / 'patches'
    if not patch_dir.exists():
        return []

    s1_pre = find_chunk_files(patch_dir, ['X_s1_pre', 'X_pre'])
    s1_flood = find_chunk_files(patch_dir, ['X_s1_flood', 'X_flood'])
    s2_pre = find_chunk_files(patch_dir, ['X_s2_pre'])
    s2_flood = find_chunk_files(patch_dir, ['X_s2_flood'])
    dem = find_chunk_files(patch_dir, ['X_dem'])
    mask = find_chunk_files(patch_dir, ['y_mask', 'Y_mask'])
    meta = sorted(patch_dir.glob('patch_metadata_*.csv'))

    ids = sorted(set(map(chunk_id_from_name, s1_pre)) & set(map(chunk_id_from_name, s1_flood)) & set(map(chunk_id_from_name, dem)) & set(map(chunk_id_from_name, mask)))
    by = lambda files: {chunk_id_from_name(f): f for f in files}
    maps = {
        's1_pre': by(s1_pre),
        's1_flood': by(s1_flood),
        's2_pre': by(s2_pre),
        's2_flood': by(s2_flood),
        'dem': by(dem),
        'mask': by(mask),
        'meta': by(meta),
    }

    chunks = []
    for cid in ids:
        chunks.append({
            'year': year,
            'chunk_id': cid,
            's1_pre': maps['s1_pre'][cid],
            's1_flood': maps['s1_flood'][cid],
            's2_pre': maps['s2_pre'].get(cid),
            's2_flood': maps['s2_flood'].get(cid),
            'dem': maps['dem'][cid],
            'mask': maps['mask'][cid],
            'meta': maps['meta'].get(cid),
        })
    return chunks


def summarize_chw_batch(arr):
    # arr shape: N,C,H,W -> N,(C*4) using mean/std/min/max.
    arr = np.asarray(arr, dtype=np.float32)
    means = arr.mean(axis=(2, 3))
    stds = arr.std(axis=(2, 3))
    mins = arr.min(axis=(2, 3))
    maxs = arr.max(axis=(2, 3))
    return np.concatenate([means, stds, mins, maxs], axis=1).astype(np.float32)


def summarize_dem_batch(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    return summarize_chw_batch(arr)


def load_optional_array(path, reference_n, channels=0):
    if path is None or not Path(path).exists():
        return None
    arr = np.load(path, mmap_mode='r')
    if len(arr) != reference_n:
        print('Warning: optional array length mismatch:', path, len(arr), reference_n)
        return None
    return arr


## 5. Hydrological Sequence Functions

For each state/year, this builds one 30-day rainfall/river sequence. During training, each node can use the sequence for its year.


In [ ]:
def normalize_columns(df):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def find_date_col(df):
    candidates = ['Date', 'date', 'Data Acquisition Time', 'data acquisition time', 'datetime', 'Time']
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        if 'date' in c.lower() or 'time' in c.lower():
            return c
    return None


def find_rain_col(df):
    candidates = ['Avg_rainfall', 'rainfall_mm', 'Rainfall', 'rainfall', 'avg_rainfall']
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if 'rain' in lc:
            return c
    return None


def find_river_col(df):
    for c in df.columns:
        lc = c.lower()
        if 'river water level' in lc or 'water level' in lc or 'river_level' in lc:
            return c
    for c in df.columns:
        lc = c.lower()
        if 'discharge' in lc or 'river water discharge' in lc:
            return c
    return None


def load_rainfall_daily(files):
    frames = []
    for path in files:
        if not Path(path).exists():
            continue
        df = normalize_columns(pd.read_csv(path))
        date_col = find_date_col(df)
        rain_col = find_rain_col(df)
        if date_col is None or rain_col is None:
            print('Skipping rainfall file with unknown schema:', path)
            continue
        df['date'] = pd.to_datetime(df[date_col], errors='coerce').dt.floor('D')
        df['rain'] = pd.to_numeric(df[rain_col], errors='coerce')
        df = df.dropna(subset=['date', 'rain'])
        frames.append(df[['date', 'rain']])
    if not frames:
        return pd.DataFrame(columns=['date', 'rain_mean', 'rain_sum'])
    all_df = pd.concat(frames, ignore_index=True)
    return all_df.groupby('date', as_index=False).agg(rain_mean=('rain', 'mean'), rain_sum=('rain', 'sum'))


def load_river_daily(files):
    frames = []
    for path in files:
        if not Path(path).exists():
            continue
        header = pd.read_csv(path, nrows=0).columns
        df = normalize_columns(pd.read_csv(path))
        date_col = find_date_col(df)
        river_col = find_river_col(df)
        if date_col is None or river_col is None:
            print('Skipping river file with unknown schema:', path)
            continue
        df['date'] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True).dt.floor('D')
        df['river'] = pd.to_numeric(df[river_col], errors='coerce')
        df = df.dropna(subset=['date', 'river'])
        frames.append(df[['date', 'river']])
    if not frames:
        return pd.DataFrame(columns=['date', 'river_mean', 'river_max'])
    all_df = pd.concat(frames, ignore_index=True)
    return all_df.groupby('date', as_index=False).agg(river_mean=('river', 'mean'), river_max=('river', 'max'))


SEQ_FEATURE_COLS = ['rain_mean', 'rain_sum', 'river_mean', 'river_max']


def build_sequences_by_year(state, years=YEARS, seq_len=SEQ_LEN):
    config = STATE_CONFIGS[state]
    rain_daily = load_rainfall_daily(expand_drive_candidates(config['rainfall_files']))
    river_daily = load_river_daily(expand_drive_candidates(config['river_files']))
    if len(rain_daily):
        rain_daily = rain_daily.set_index('date')
    else:
        rain_daily = pd.DataFrame(columns=['rain_mean', 'rain_sum'])
    if len(river_daily):
        river_daily = river_daily.set_index('date')
    else:
        river_daily = pd.DataFrame(columns=['river_mean', 'river_max'])

    sequences = {}
    for year in years:
        # Use the last SEQ_LEN days before/around peak monsoon window as a deterministic sequence.
        end_date = pd.Timestamp(f'{year}-08-31')
        dates = pd.date_range(end=end_date, periods=seq_len, freq='D')
        seq_df = pd.DataFrame(index=dates)
        seq_df = seq_df.join(rain_daily, how='left').join(river_daily, how='left')
        seq_df = seq_df[SEQ_FEATURE_COLS].fillna(0.0)
        sequences[year] = seq_df.values.astype(np.float32)
    return sequences


## 6. Graph Construction

Nodes are patches. Edges connect neighboring patches within the same state and same year. No cross-state graph is created.


In [ ]:
def build_edges_from_metadata(meta_df, neighbor_radius=NEIGHBOR_RADIUS):
    required = {'year', 'grid_row', 'grid_col'}
    if not required.issubset(meta_df.columns):
        # Fallback names from Pipeline 1 metadata.
        rename = {}
        if 'row' in meta_df.columns and 'grid_row' not in meta_df.columns:
            rename['row'] = 'grid_row'
        if 'col' in meta_df.columns and 'grid_col' not in meta_df.columns:
            rename['col'] = 'grid_col'
        meta_df = meta_df.rename(columns=rename)
    if not required.issubset(meta_df.columns):
        raise ValueError(f'Metadata must include year/grid_row/grid_col or year/row/col. Found: {list(meta_df.columns)}')

    edges = []
    for year, group in meta_df.groupby('year'):
        lookup = {(int(r.grid_row), int(r.grid_col)): int(idx) for idx, r in group.iterrows()}
        for idx, row in group.iterrows():
            r0 = int(row.grid_row)
            c0 = int(row.grid_col)
            for dr in range(-neighbor_radius, neighbor_radius + 1):
                for dc in range(-neighbor_radius, neighbor_radius + 1):
                    if dr == 0 and dc == 0:
                        continue
                    other = lookup.get((r0 + dr, c0 + dc))
                    if other is not None:
                        edges.append((int(idx), other))
    if not edges:
        return np.empty((2, 0), dtype=np.int64)
    return np.array(edges, dtype=np.int64).T


## 7. Per-State Graph Dataset Builder

This is resumable. If a state's `manifest.json` already exists and `FORCE_REBUILD=False`, the state is skipped.


In [ ]:
FORCE_REBUILD = False
SCALE_NODE_FEATURES = True


def process_state(state):
    print('\n' + '=' * 80)
    print('Processing state:', state)
    patch_root = resolve_patch_root(state)
    if patch_root is None:
        print(state, 'skipped: no Pipeline 1 patch root found')
        return None

    state_out = OUTPUT_ROOT / state.lower().replace(' ', '_')
    state_out.mkdir(parents=True, exist_ok=True)
    manifest_path = state_out / 'manifest.json'
    if manifest_path.exists() and not FORCE_REBUILD:
        print(state, 'already processed. Set FORCE_REBUILD=True to rebuild.')
        return json.loads(manifest_path.read_text())

    feature_chunks = []
    target_chunks = []
    binary_chunks = []
    meta_frames = []
    chunk_summary = []

    global_offset = 0
    for year in YEARS:
        chunks = collect_year_chunks(patch_root, year)
        print(f'{state} {year}: chunks found = {len(chunks)}')
        for info in tqdm(chunks, desc=f'{state} {year}'):
            s1_pre = np.load(info['s1_pre'], mmap_mode='r')
            s1_flood = np.load(info['s1_flood'], mmap_mode='r')
            dem = np.load(info['dem'], mmap_mode='r')
            mask = np.load(info['mask'], mmap_mode='r')
            n = len(mask)

            s2_pre = load_optional_array(info['s2_pre'], n)
            s2_flood = load_optional_array(info['s2_flood'], n)

            parts = [
                summarize_chw_batch(s1_pre),
                summarize_chw_batch(s1_flood),
                summarize_chw_batch(np.asarray(s1_flood, dtype=np.float32) - np.asarray(s1_pre, dtype=np.float32)),
                summarize_dem_batch(dem),
            ]
            if s2_pre is not None and s2_flood is not None:
                parts.extend([
                    summarize_chw_batch(s2_pre),
                    summarize_chw_batch(s2_flood),
                    summarize_chw_batch(np.asarray(s2_flood, dtype=np.float32) - np.asarray(s2_pre, dtype=np.float32)),
                ])
            features = np.concatenate(parts, axis=1).astype(np.float32)

            y_fraction = np.asarray(mask, dtype=np.float32).mean(axis=tuple(range(1, mask.ndim))).astype(np.float32)
            y_binary = (y_fraction >= FLOOD_THRESHOLD).astype(np.int64)

            if info['meta'] is not None and Path(info['meta']).exists():
                meta = pd.read_csv(info['meta'])
            else:
                meta = pd.DataFrame({'row': np.arange(n), 'col': np.zeros(n, dtype=int)})
            meta = meta.copy()
            if 'year' not in meta.columns:
                meta['year'] = year
            if 'chunk_id' not in meta.columns:
                meta['chunk_id'] = info['chunk_id']
            if 'row' in meta.columns and 'grid_row' not in meta.columns:
                meta['grid_row'] = meta['row']
            if 'col' in meta.columns and 'grid_col' not in meta.columns:
                meta['grid_col'] = meta['col']
            if 'grid_row' not in meta.columns:
                meta['grid_row'] = np.arange(n)
            if 'grid_col' not in meta.columns:
                meta['grid_col'] = 0
            meta['state'] = state
            meta['local_node_id'] = np.arange(n)
            meta['node_id'] = np.arange(global_offset, global_offset + n)
            meta['source_chunk'] = info['chunk_id']
            meta['target_fraction'] = y_fraction
            meta['target_binary'] = y_binary
            global_offset += n

            feature_chunks.append(features)
            target_chunks.append(y_fraction)
            binary_chunks.append(y_binary)
            meta_frames.append(meta)
            chunk_summary.append({'year': year, 'chunk_id': info['chunk_id'], 'nodes': int(n), 'feature_dim': int(features.shape[1])})

            del s1_pre, s1_flood, dem, mask, s2_pre, s2_flood, features, y_fraction, y_binary, meta
            gc.collect()

    if not feature_chunks:
        print(state, 'skipped: no patch chunks were processed')
        return None

    X = np.concatenate(feature_chunks, axis=0).astype(np.float32)
    y_fraction = np.concatenate(target_chunks, axis=0).astype(np.float32)
    y_binary = np.concatenate(binary_chunks, axis=0).astype(np.int64)
    meta_df = pd.concat(meta_frames, ignore_index=True)
    meta_df.index = np.arange(len(meta_df))

    if SCALE_NODE_FEATURES:
        scaler = StandardScaler()
        X = scaler.fit_transform(X).astype(np.float32)
        np.save(state_out / 'node_feature_mean.npy', scaler.mean_.astype(np.float32))
        np.save(state_out / 'node_feature_scale.npy', scaler.scale_.astype(np.float32))

    edge_index = build_edges_from_metadata(meta_df, neighbor_radius=NEIGHBOR_RADIUS)
    sequences = build_sequences_by_year(state)
    sequence_years = np.array(sorted(sequences), dtype=np.int64)
    sequence_tensor = np.stack([sequences[int(y)] for y in sequence_years], axis=0).astype(np.float32)
    year_to_seq_idx = {int(y): i for i, y in enumerate(sequence_years)}
    node_sequence_index = meta_df['year'].map(year_to_seq_idx).fillna(-1).astype(np.int64).values

    np.save(state_out / 'node_features.npy', X)
    np.save(state_out / 'target_fraction.npy', y_fraction)
    np.save(state_out / 'target_binary.npy', y_binary)
    np.save(state_out / 'edge_index.npy', edge_index)
    np.save(state_out / 'sequence_years.npy', sequence_years)
    np.save(state_out / 'hydro_sequences.npy', sequence_tensor)
    np.save(state_out / 'node_sequence_index.npy', node_sequence_index)
    meta_df.to_csv(state_out / 'node_metadata.csv', index=False)
    pd.DataFrame(chunk_summary).to_csv(state_out / 'chunk_summary.csv', index=False)

    manifest = {
        'state': state,
        'patch_root': str(patch_root),
        'output_dir': str(state_out),
        'nodes': int(X.shape[0]),
        'node_feature_dim': int(X.shape[1]),
        'edges': int(edge_index.shape[1]),
        'target': 'target_fraction and target_binary',
        'flood_threshold': float(FLOOD_THRESHOLD),
        'sequence_length': int(SEQ_LEN),
        'sequence_feature_cols': SEQ_FEATURE_COLS,
        'years': [int(y) for y in sequence_years.tolist()],
        'neighbor_radius': int(NEIGHBOR_RADIUS),
        'files': {
            'node_features': 'node_features.npy',
            'target_fraction': 'target_fraction.npy',
            'target_binary': 'target_binary.npy',
            'edge_index': 'edge_index.npy',
            'node_metadata': 'node_metadata.csv',
            'hydro_sequences': 'hydro_sequences.npy',
            'sequence_years': 'sequence_years.npy',
            'node_sequence_index': 'node_sequence_index.npy',
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Saved graph dataset:', state_out)
    print(json.dumps({k: manifest[k] for k in ['state', 'nodes', 'node_feature_dim', 'edges']}, indent=2))

    del X, y_fraction, y_binary, edge_index, sequence_tensor, meta_df
    gc.collect()
    return manifest


## 8. Run Sequentially: Assam -> Kerala -> Uttarpradesh -> Odisha

This cell intentionally processes one state at a time to stay within Colab Free memory limits. If it stops midway, rerun the notebook; completed states are skipped unless `FORCE_REBUILD=True`.


In [ ]:
manifests = []
for state in STATE_ORDER:
    manifest = process_state(state)
    if manifest is not None:
        manifests.append(manifest)

summary = pd.DataFrame([
    {
        'state': m['state'],
        'nodes': m['nodes'],
        'feature_dim': m['node_feature_dim'],
        'edges': m['edges'],
        'output_dir': m['output_dir'],
    }
    for m in manifests
])
summary.to_csv(OUTPUT_ROOT / 'pipeline2_preprocessing_summary.csv', index=False)
display(summary)
print('Pipeline 2 preprocessing complete. Summary:', OUTPUT_ROOT / 'pipeline2_preprocessing_summary.csv')


## 9. Verify Saved Graph Datasets


In [ ]:
for state in STATE_ORDER:
    state_dir = OUTPUT_ROOT / state.lower().replace(' ', '_')
    print('\n', state, state_dir)
    manifest_path = state_dir / 'manifest.json'
    if not manifest_path.exists():
        print('  not processed')
        continue
    manifest = json.loads(manifest_path.read_text())
    print(' manifest:', {k: manifest[k] for k in ['nodes', 'node_feature_dim', 'edges']})
    for name in ['node_features.npy', 'target_fraction.npy', 'target_binary.npy', 'edge_index.npy', 'hydro_sequences.npy', 'node_sequence_index.npy', 'node_metadata.csv']:
        p = state_dir / name
        if not p.exists():
            print('  missing', name)
            continue
        if p.suffix == '.npy':
            arr = np.load(p, mmap_mode='r')
            print(' ', name, arr.shape, arr.dtype)
        else:
            print(' ', name, pd.read_csv(p, nrows=3).shape, 'preview rows')


## 10. Output Contract for the Future Training Notebook

Each state folder is independent and contains:

```text
node_features.npy          # N x F node feature matrix
edge_index.npy             # 2 x E graph edges
target_fraction.npy        # N flood fraction target
target_binary.npy          # N binary flood label
node_metadata.csv          # state/year/grid row/col/node ids
hydro_sequences.npy        # Y x SEQ_LEN x 4 rainfall/river sequences
sequence_years.npy         # years corresponding to hydro_sequences
node_sequence_index.npy    # maps each node to a sequence row
manifest.json              # dataset contract
```

The future GNN+GRU notebook should load one state at a time or explicitly perform cross-state experiments after preprocessing. This notebook does not merge states.
